In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
import time
import numpy as np
from pyspark.sql.functions import when, col, abs


StatementMeta(, 11641988-611c-41de-9f86-7435a39c18b1, 7, Finished, Available, Finished, False)

GMV

In [ ]:
def df_gmv():
    query_gmv = """
        SELECT
            domainId,
            SUM(summary_total) / 3.0 AS MediaGMV
        FROM
            MongoDB_Pedidos_Geral
        WHERE
            status_canceled_isCanceled = 0
            AND isClosed = 1
            AND settings_createdAt_TIMESTAMP >= add_months(trunc(current_date(), 'MM'), -3)
            AND settings_createdAt_TIMESTAMP < trunc(current_date(), 'MM')
        GROUP BY
            domainId
    """
    df = spark.sql(query_gmv)
    return df

StatementMeta(, 11641988-611c-41de-9f86-7435a39c18b1, 8, Finished, Available, Finished, False)

TPV Vestipago Cartão e Pix

In [ ]:
def df_tpv():
    date_filter = """
        AND settings_createdAt_TIMESTAMP >= add_months(trunc(current_date(), 'MM'), -3)
        AND settings_createdAt_TIMESTAMP < trunc(current_date(), 'MM')
    """

    query_cartao = f"""
        SELECT
            domainId as DominioId,
            SUM(payment_transaction_value) / 3.0 AS TPV_Cartao
        FROM
            MongoDB_Pedidos_Geral
        WHERE
            status_canceled_isCanceled = 0
            AND isClosed = 1
            {date_filter}
            AND payment_method = 'CREDIT_CARD'
            AND payment_isPaid = 1
        GROUP BY
            domainId
    """

    query_PIX = f"""
        SELECT
            domainId as domainId2,
            SUM(payment_transaction_value) / 3.0 AS TPV_Pix
        FROM
            MongoDB_Pedidos_Geral
        WHERE
            status_canceled_isCanceled = 0
            AND isClosed = 1
            {date_filter}
            AND payment_method = 'PIX'
            AND payment_isPaid = 1
        GROUP BY
            domainId
    """

    dfCartao = spark.sql(query_cartao)
    dfPix = spark.sql(query_PIX)

    df = dfCartao.join(
        dfPix,
        dfPix['domainId2'] == dfCartao['DominioId'],
        "left"
    )

    df = df.drop('domainId2')
    
    return df

StatementMeta(, 11641988-611c-41de-9f86-7435a39c18b1, 9, Finished, Available, Finished, False)

NPS

In [ ]:
def df_nps():
    query_nps = """
        SELECT
            Dominio as domain_id,
            (
                (SUM(CASE WHEN Nota >= 9 THEN 1 ELSE 0 END) - SUM(CASE WHEN Nota <= 6 THEN 1 ELSE 0 END)) * 100.0
            ) / COUNT(*) AS NPS
        FROM
            sheets_pesquisanps
        GROUP BY
            Dominio
    """
    dfspark = spark.sql(query_nps)
    df_limpo = dfspark.withColumn(
        "NPS",
        when(abs(col("NPS")) < 1e-12, 0.0).otherwise(col("NPS"))
    )

    return df_limpo

StatementMeta(, 11641988-611c-41de-9f86-7435a39c18b1, 10, Finished, Available, Finished, False)

MRR

In [ ]:
def df_mrr():
    query_mrr = """    
        SELECT
            i.customer_id,
            i.id,
            i.due_date,
            (CASE
                WHEN i.items_price_cents IS NULL THEN i.total_cents/100
                ELSE i.items_price_cents/100
            END) as ValorFatura,
            c.custom_variables_value as Dominio2
        FROM
            iugu_invoices i
        INNER JOIN
            iugu_customers c ON (c.custom_variables_name LIKE '%domain_id%' AND c.id = i.customer_id)
        WHERE
            i.status <> 'canceled'
            -- Pega do dia 1 do mês passado até o dia 1 do mês atual (exclusivo)
            AND i.due_date_TIMESTAMP >= add_months(trunc(current_date(), 'MM'), -1)
            AND i.due_date_TIMESTAMP < trunc(current_date(), 'MM')
    """
    dfspark = spark.sql(query_mrr)
    dfspark = dfspark.drop_duplicates()

    df2 = dfspark.drop('customer_id').drop('id').drop('due_date')
    df = df2.groupBy("Dominio2").sum('ValorFatura').withColumnRenamed("sum(ValorFatura)","MRR")
    
    return df

StatementMeta(, 11641988-611c-41de-9f86-7435a39c18b1, 11, Finished, Available, Finished, False)

DOM

In [ ]:
def df_dominios():
    query_dom = """
        SELECT 
            d.id as Dominio, 
            a.name as CS, 
            pay.isEnabled as Vestipago_Ativo, 
            i.name as Integracao
        FROM 
            ODBC_Domains d
        LEFT JOIN 
            mongodb_companies pay on (pay.domainId = d.id AND pay.isEnabled = 1 AND (paymentSettings_iugu_creditCard_isEnabled = 1 OR paymentSettings_iugu_pix_isEnabled = 1))
        LEFT JOIN 
            ODBC_Angels a on a.id = d.angel_id
        LEFT JOIN 
            ODBC_Partners p on p.id = d.partner_id
        LEFT JOIN 
            ODBC_Integrations i on i.id = d.integration_id
        WHERE 
            d.modulos LIKE '%vendas%'
        AND 
            p.name NOT IN ('Trial', 'Treino')
        GROUP BY 
            d.id, 
            a.name, 
            pay.isEnabled, 
            i.name
    """
    df = spark.sql(query_dom)
    return df

StatementMeta(, 11641988-611c-41de-9f86-7435a39c18b1, 12, Finished, Available, Finished, False)

In [ ]:
gmv = df_gmv()
nps = df_nps()
mrr = df_mrr()
dom = df_dominios()
tpv = df_tpv()

df_intermediario = dom.join(
    nps,
    dom["Dominio"] == nps["domain_id"],
    "left"
)


df_intermediario2 = df_intermediario.join(
    mrr,
    df_intermediario["Dominio"] == mrr["Dominio2"],
    "left"
)

df_intermediario3 = df_intermediario2.join(
    tpv,
    df_intermediario2["Dominio"] == tpv["DominioID"],
    "left"
)

dfFinal = df_intermediario3.join(
        gmv,
        df_intermediario3["Dominio"] == gmv["domainId"],
        "left"
)
df = dfFinal.drop(*["domainId", "domain_id", "Dominio2", "DominioID"])

df.show()

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Capivara")

StatementMeta(, 11641988-611c-41de-9f86-7435a39c18b1, 13, Finished, Available, Finished, False)

+-------+----------------+---------------+------------------+-----+------+------------------+------------------+------------------+
|Dominio|              CS|Vestipago_Ativo|        Integracao|  NPS|   MRR|        TPV_Cartao|           TPV_Pix|          MediaGMV|
+-------+----------------+---------------+------------------+-----+------+------------------+------------------+------------------+
|1516172|Thamiris Ribeiro|           true|               API| NULL|1302.0|              NULL|              NULL| 34670.87333333333|
| 386662|   Elisa Marques|           true|               API| NULL| 261.0|            367.24|              NULL|            4345.0|
|1746701| Jennyfer Rabelo|           true|              Tiny| NULL| 429.0|          25208.11|          29130.87|59381.616666666676|
|1232863|Thamiris Ribeiro|           true|              Tiny| NULL|1155.0| 7035.896666666667|          16956.59|          33416.53|
|1436061| Jennyfer Rabelo|           true|          Bling V3| NULL| 317.0|  

In [ ]:
display(df)

StatementMeta(, 11641988-611c-41de-9f86-7435a39c18b1, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f0558c7b-1d23-4217-b501-de8c856b6502)